# 06 — Attention Kernels

Covers Phase 6 of the kernel roadmap — the main goal:
- `naive_attention` — O(N²) memory, baseline
- `sdpa` — scaled dot-product + causal masking
- `multi_head_attention` — head splitting
- `flash_attention_v1` — online softmax, O(N) memory
- `flash_attention_v2` — Q-block parallelism, fewer HBM writes

**Metric**: TFLOPS — `(4 × N² × d × 1e-12) / (ms × 1e-3)`

**References**:
- FlashAttention: https://arxiv.org/abs/2205.14135
- FlashAttention-2: https://arxiv.org/abs/2307.08691

In [ ]:
# ── Setup ────────────────────────────────────────────────────────────────────
import os
REPO_DIR = "/content/triton-kernels"
if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone https://github.com/YOUR_USERNAME/triton-kernels.git {REPO_DIR}
os.chdir(REPO_DIR)
!bash scripts/setup_colab.sh
import torch
assert torch.cuda.is_available()
print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import torch, triton

# Uncomment as kernels are implemented:
# from kernels.attention.naive_attention import naive_attention, test_naive_attention, benchmark_naive_attention
# from kernels.attention.sdpa import sdpa, test_sdpa, benchmark_sdpa
# from kernels.attention.multi_head_attention import multi_head_attention, test_multi_head_attention, benchmark_multi_head_attention
# from kernels.attention.flash_attention_v1 import flash_attention_v1, test_flash_attention_v1, benchmark_flash_attention_v1
# from kernels.attention.flash_attention_v2 import flash_attention_v2, test_flash_attention_v2, benchmark_flash_attention_v2

print("Imports ready")

## 1. naive_attention

**File**: `kernels/attention/naive_attention.py`  
**Formula**: `softmax(Q K^T / sqrt(d)) V`  
**Memory**: O(N²) — materializes the full attention matrix in HBM  
**Serves as**: correctness reference for all subsequent attention kernels

In [ ]:
# ── naive_attention: Correctness ─────────────────────────────────────────────
# test_naive_attention()

In [ ]:
# ── naive_attention: Benchmark ───────────────────────────────────────────────
# benchmark_naive_attention.run(print_data=True, show_plots=True,
#     save_path="benchmarks/results/attention/naive_attention_benchmark.png")

**Interpretation**: Add notes here.

## 2. sdpa

**File**: `kernels/attention/sdpa.py`  
**Adds**: Causal masking (decoder attention)

In [ ]:
# ── sdpa: Correctness ────────────────────────────────────────────────────────
# test_sdpa()

In [ ]:
# ── sdpa: Benchmark ──────────────────────────────────────────────────────────
# benchmark_sdpa.run(print_data=True, show_plots=True,
#     save_path="benchmarks/results/attention/sdpa_benchmark.png")

**Interpretation**: Add notes here.

## 3. multi_head_attention

**File**: `kernels/attention/multi_head_attention.py`  
**Adds**: Head splitting via `tl.program_id(1)` for the head axis

In [ ]:
# ── multi_head_attention: Correctness ────────────────────────────────────────
# test_multi_head_attention()

In [ ]:
# ── multi_head_attention: Benchmark ──────────────────────────────────────────
# benchmark_multi_head_attention.run(print_data=True, show_plots=True,
#     save_path="benchmarks/results/attention/multi_head_attention_benchmark.png")

**Interpretation**: Add notes here.

## 4. flash_attention_v1

**File**: `kernels/attention/flash_attention_v1.py`  
**Key**: Online softmax trick — accumulates max/sum across K,V tiles so the full N×N matrix never materializes. O(N) memory.
**Reference**: Dao et al. 2022 — https://arxiv.org/abs/2205.14135

In [ ]:
# ── flash_attention_v1: Correctness ──────────────────────────────────────────
# test_flash_attention_v1()

In [ ]:
# ── flash_attention_v1: Benchmark ────────────────────────────────────────────
# benchmark_flash_attention_v1.run(print_data=True, show_plots=True,
#     save_path="benchmarks/results/attention/flash_attention_v1_benchmark.png")

**Interpretation**: Add notes here.

## 5. flash_attention_v2

**File**: `kernels/attention/flash_attention_v2.py`  
**Key improvements over v1**:
- Parallelism over Q blocks (not just K,V)
- Fewer HBM writes
- Better occupancy via rearranged outer loop

**Reference**: Dao 2023 — https://arxiv.org/abs/2307.08691

In [ ]:
# ── flash_attention_v2: Correctness ──────────────────────────────────────────
# test_flash_attention_v2()

In [ ]:
# ── flash_attention_v2: Benchmark ────────────────────────────────────────────
# benchmark_flash_attention_v2.run(print_data=True, show_plots=True,
#     save_path="benchmarks/results/attention/flash_attention_v2_benchmark.png")

**Interpretation**: Add notes here.

In [ ]:
# ── Summary Table ────────────────────────────────────────────────────────────
# import pandas as pd, glob
# csvs = glob.glob("benchmarks/results/attention/*.csv")
# if csvs:
#     print(pd.concat([pd.read_csv(f) for f in csvs], ignore_index=True).to_string(index=False))
# else:
#     print("No CSVs yet.")